In [ ]:
import torch
import matplotlib.pyplot as plt

!pip install benchmarkfcns==2.5.0
from benchmarkfcns import brent

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 47.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
# brent (2D)

def bre(x):
  # function
  dim = x.shape[0]
  assert x.shape == (dim,1)
  y = (x[0] + 10)**2 + (x[1] + 10)**2 + torch.exp(-x[0]**2 - x[1]**2);

  # gradient
  # X = 2*(x[0] + 10)+torch.exp(-x[0]**2 - x[1]**2)*(-2*x[0])
  # Y = 2*(x[1] + 10)+torch.exp(-x[0]**2 - x[1]**2)*(-2*x[1])
  # dy = torch.tensor([[X],[Y]])

  # Check that this is true (we set this before the for loop)
  assert x.requires_grad == True
  # Make sure output is as scalar
  assert y.shape == (1,1)
  grad_term = torch.autograd.grad(outputs=y, inputs=x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

  # Save space (and get rid of any weird extra stuff)
  y = y.detach()
  x = x.detach()
  #grad_term = grad_term.detach()


  # Make sure the gradient is the same sizeas the input
  assert grad_term.shape == x.shape

  # Return
  return grad_term

  assert dy.shape == (x.shape)

  return y, dy


# verify function implementation

  for i in range(1000):
    x = torch.randn(2,1)
    y = bre(x)[0]
    x = x.view(1,2).numpy()
    y2 = brent(x)
    assert abs(y.item()-y2.item())<1e-1

In [ ]:
# define gradient descent
def gradient_descent(ack2, step_size, x0, tol=1e-3, max_iters=100):

  x_hist = [] # save x at each iteration for plotting
  f_hist = [] # save f values at each iteration for plotting
  grad_norm_hist = [] # save gradient norm at each iteration for plotting

  x = x0
  x.requires_grad = True

  # store values at 0th iteration
  x_hist.append(x0)
  fx = ack2(x0)[0]
  f_hist.append(fx)
  grad_fx = ack2(x0)[1]
  grad_norm = torch.norm(grad_fx)
  grad_norm_hist.append(grad_norm)

  for i in range(max_iters):

    # gradient descent update
    x = x - step_size * grad_fx
    grad_fx = ack2(x)[1]
    fx = ack2(x)[0]
    # change stepsize
    # if torch.norm(grad_fx)-grad_norm <= tol:
    #   step_size = step_size/2

    # evaluate for plotting
    grad_norm = torch.norm(grad_fx)

    # store values
    x_hist.append(x.detach())
    f_hist.append(fx.detach())
    grad_norm_hist.append(grad_norm.detach())

    # print values
    print('i: %d, fx: %.15e, grad_norm: %.15e' % (i, fx.item(), grad_norm.item()))

    # check stopping criterion
    if grad_norm <= tol: # or grad_norm == torch.inf:
      print('converged!')
      break

  x_opt = x.detach()
  return x_opt, f_hist, grad_norm_hist, x_hist

In [ ]:
# set up optimization parameters

dim = 2
# step_size = 9.97e-3
max_iters = 1000
x0 = torch.rand(dim,1)
tol = 1e-3


step_size = 1e-1
x0_1 = 5*torch.rand(dim,1)
x_opt1, f_hist1, grad_norm_hist1, x_hist1 = gradient_descent(bre, step_size, x0_1, tol=tol, max_iters=max_iters)


i: 0, fx: 2.026170501708984e+02, grad_norm: 2.833983039855957e+01
i: 1, fx: 1.295659027099609e+02, grad_norm: 2.276683616638184e+01
i: 2, fx: 8.291935729980469e+01, grad_norm: 1.821201324462891e+01
i: 3, fx: 5.306838989257812e+01, grad_norm: 1.456961059570312e+01
i: 4, fx: 3.396376800537109e+01, grad_norm: 1.165568828582764e+01
i: 5, fx: 2.173680877685547e+01, grad_norm: 9.324549674987793e+00
i: 6, fx: 1.391155910491943e+01, grad_norm: 7.459640502929688e+00
i: 7, fx: 8.903396606445312e+00, grad_norm: 5.967711925506592e+00
i: 8, fx: 5.698176383972168e+00, grad_norm: 4.774170875549316e+00
i: 9, fx: 3.646833419799805e+00, grad_norm: 3.819336891174316e+00
i: 10, fx: 2.333973646163940e+00, grad_norm: 3.055469512939453e+00
i: 11, fx: 1.493743538856506e+00, grad_norm: 2.444375991821289e+00
i: 12, fx: 9.559957981109619e-01, grad_norm: 1.955500721931458e+00
i: 13, fx: 6.118373870849609e-01, grad_norm: 1.564400672912598e+00
i: 14, fx: 3.915753960609436e-01, grad_norm: 1.251519799232483e+00
i: 15

In [ ]:
# plot solution paths
# create 2D grid from -5 to 5

# define range and number of pts
xx = torch.linspace(-5,5,100)
yy = torch.linspace(-5,5,100)

# creat grid
X, Y = torch.meshgrid(xx, yy, indexing = 'xy') # 'xy' ensures correct indexing for 2D grid

Z = torch.zeros(X.shape)
for i in range(100):
  for j in range(100):
    input_vec = torch.tensor([X[i,j], Y[i,j]]).view(2,1)
    # put in function
    Z[i,j] = bre(input_vec)[0].item()

In [ ]:
# plot fn val vs itr
plt.figure(figsize=(8, 6))
plt.plot([f.detach().numpy() for f in f_hist1])
plt.xlabel("Iteration")
plt.ylabel("Function Value (brent)")
plt.title("Function Value vs. Iteration")
plt.savefig("brent.fn_itr.png")  # Save the plot
plt.show()

# plot grad norm vs itr
plt.figure(figsize=(8,6))
plt.plot(grad_norm_hist1)
plt.xlabel("Iteration")
plt.ylabel("Grad. Norm")
plt.title("Grad. Norm vs. Iteration")
plt.savefig("brent.grad_itr.png")  # Save the plot
plt.show()

# plot gradient trajectory
plt.figure(figsize=(8,6))

# plots the landscape
plt.contour(X, Y, Z, levels=30, cmap='viridis')

x_coords = [x[0].item() for x in x_hist1] # extract x-coordinates
y_coords = [x[1].item() for x in x_hist1] # extract y-coordinates

# plot GD trajectory
plt.plot(x_coords, y_coords, marker='o', color='blue', label='GD Traj 1')

# plot minimizer
plt.plot(0,0, marker='x', color='red', markersize=10, markeredgewidth=5)

plt.colorbar(label='f(x)')
plt.title('GD Trajectories')
plt.xlabel('x-axis')
plt.ylabel('y-axis')
plt.savefig("brent.trjectory.png")  # Save the plot
plt.show()